In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from matplotlib import pyplot as plt
import pandas as pd
import pickle

df = pd.read_csv("Driver_Behavior.csv")

print(df.head())

X = df.drop('behavior_label', axis='columns')

print(X.head())

plt.figure(figsize=(8,6))

plt.scatter(X['speed_kmph'], X['throttle'])

plt.xlabel('Speed (kmph)')
plt.ylabel('Throttle')

plt.show()

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(X_scaled.head())

km = KMeans(n_clusters=3, random_state=42)

y_predicted = km.fit_predict(X_scaled)

X_scaled['cluster'] = y_predicted

print(X_scaled.head())

print(km.cluster_centers_)

df1 = X_scaled[X_scaled.cluster == 0]
df2 = X_scaled[X_scaled.cluster == 1]
df3 = X_scaled[X_scaled.cluster == 2]

plt.figure(figsize=(8,6))

plt.scatter(df1['speed_kmph'], df1['throttle'], label='Cluster 0')
plt.scatter(df2['speed_kmph'], df2['throttle'], label='Cluster 1')
plt.scatter(df3['speed_kmph'], df3['throttle'], label='Cluster 2')

plt.scatter(
    km.cluster_centers_[:, 0],
    km.cluster_centers_[:, 5],
    marker='*',
    s=300,
    label='Centroids'
)

plt.xlabel('Speed (Scaled)')
plt.ylabel('Throttle (Scaled)')

plt.legend()

plt.show()

sse = []

k_rng = range(1, 11)

for k in k_rng:
    km_test = KMeans(n_clusters=k, random_state=42)

    km_test.fit(X_scaled.drop('cluster', axis='columns'))

    sse.append(km_test.inertia_)

plt.figure(figsize=(8,6))

plt.xlabel('K')
plt.ylabel('Sum of Squared Error')

plt.plot(k_rng, sse)

plt.show()

sample_driver = [[
    40,
    0.5,
    0.3,
    20,
    -5,
    45,
    0.8,
    1,
    18,
    1.2
]]

sample_scaled = scaler.transform(sample_driver)

prediction = km.predict(sample_scaled)

print("Cluster Prediction:", prediction[0])

with open("driver_behavior_kmeans_model.pkl", "wb") as file:
    pickle.dump(km, file)

print("Model saved successfully")

with open("driver_behavior_kmeans_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

print("Model loaded successfully")

loaded_prediction = loaded_model.predict(sample_scaled)

print("Prediction using Loaded Model:", loaded_prediction[0])